Assignment 2 - Part 1 | Sentiment Classificantion with Neural Language Models | Brodie Watson | CYSE 650

The model i built is a deep averaging network or DAN, which is a simple and lightweight neural netowrk for text classification. It basically works by converting words in a review into numbers using a custom vocabulary, turning those numbers into trainable word vectors or embeddings, and then from there it would average all the word vectors together into a single summary vector for the entire review. This summary will then be passed through a hidden neural layer to predict whether the review is positive or negative. I ntentionally chose this approach over heavier models because the datasets provided is very small with only 240 reviews. A heavier model would easily memorize the small data and overfit. Whereas my model, DAN willl train in seconds on a basic CPU, and avoids overfitting, and yeilds better accuracy on unseen test data while still functioning as a complete end to end neural network.

In [93]:
from pathlib import Path #this is basically allow us to work with file and folder locations easily so that there is no version confusion or loss of where we're running from
import json # lets us read and write json data
import random #this is a built in number generator
import re # regular expressions 
from collections import Counter # counter its a dictionary subclass that counts how many times each word appears

import numpy as np
import pandas as pd #lets us work with tables and data works within rows and columns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report # these three tools help us check how good ur model's predicitons will be

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [94]:
# The supplied files are in assignment2/train.csv and assignment2/public_test.csv.
# This code finds them automatically even if the notebook is named Untitled-1.ipynb.

def find_data_folder(start_folder):
    likely_folders = [
        start_folder,
        start_folder / "assignment2",
        start_folder / "data",
    ]
    for folder in likely_folders:
        if (folder / "train.csv").exists() and (folder / "public_test.csv").exists():
            return folder
# if none of the likely folders worked then we can fall back to a recursive search. it will look in every subfolder for a file literally named train.csv
    for train_file in start_folder.rglob("train.csv"):
        folder = train_file.parent
        if (folder / "public_test.csv").exists():
            return folder
#if we get here then we were truly nable to find the data. will reply with a clear error message
    raise FileNotFoundError(
        "Could not find train.csv and public_test.csv. "
        "Set DATA_DIR manually to the folder containing both files."
    )

DATA_DIR = find_data_folder(Path.cwd())
ROOT = Path.cwd()
CHECKPOINT_DIR = ROOT / "model_checkpoint"

print("Data folder:", DATA_DIR.resolve())
print("Output folder:", ROOT.resolve())

Data folder: F:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main\assignment2
Output folder: F:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main


In [95]:
# load both csvs into pandas DataFrames which is basically a table kind of like a spreadsheet with named columns
train = pd.read_csv(DATA_DIR / "train.csv")
public_test = pd.read_csv(DATA_DIR / "public_test.csv")

#we are puling out the columns we need. x is our input data and y is the target or label we are trying to predict which is generally 0 or 1
x_train = train["text"].fillna("")
y_train = train["label"].astype(int)
x_public = public_test["text"].fillna("")
y_public = public_test["label"].astype(int)

print("Training Reviews:", len(train))
print("Training Class Counts:")
print(y_train.value_counts().sort_index().rename(index={0: "Negative", 1: "Positive"}))
print("Public Test Reviews:", len(public_test))
print("Public Test Class Counts:")
print(y_public.value_counts().sort_index().rename(index={0: "Negative", 1: "Positive"}))

Training Reviews: 240
Training Class Counts:
label
Negative     60
Positive    180
Name: count, dtype: int64
Public Test Reviews: 400
Public Test Class Counts:
label
Negative    200
Positive    200
Name: count, dtype: int64


In [96]:
# a regular expression 
TOKEN_RE = re.compile(r"[a-z']+")

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

MIN_FREQ = 2  # any words that appear fewer than this many times in training collapse into unknown

def build_vocab(texts, min_freq=MIN_FREQ):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def encode(text, vocab):
    tokens = tokenize(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    if len(ids) == 0:
        ids = [vocab["<unk>"]]
    return ids

In [97]:
#  this will wrap a text column as token id sequences. basically it says given a row number (idx) and then hand back that reviews data ready for the model
class ReviewDataset(Dataset):
    #renumbers rows 0,1,2 and so on. This is important because after a train/val split, the original pandas rows numbers are no longer consecutive
    def __init__(self, texts, labels, vocab):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

# lets pythons datset (idx) work. this is called once per row, per epoch
    def __getitem__(self, idx):
        ids = encode(self.texts.iloc[idx], self.vocab)
        label = int(self.labels.iloc[idx]) if self.labels is not None else -1
        return ids, label

#this pads each batch to the longest review in that batch with no fixed max length
def collate(batch):
    seqs, labels = zip(*batch)
    max_len = max(len(s) for s in seqs)
    x = torch.zeros(len(seqs), max_len, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.long)
    return x, y

In [ ]:
class NeuralClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=150, hidden_dim=128, num_classes=2, pad_idx=0, dropout=0.6):
        super().__init__()
        # convert word IDs into vector representations
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        #first fully connected layer from embedding size to hidden size
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        # dropout layer to help prevent overfitting
        self.dropout = nn.Dropout(dropout)
        # our output layer hidden size to number of classes
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        #create a mask to mark non-padding tokens
        mask = (x != 0).unsqueeze(-1).float()
        # get word embedding and zero out any padding positions          # ignore <pad> tokens
        emb = self.embedding(x) * mask
        # Add up word embeddings across each sentence
        summed = emb.sum(dim=1)
        # count non padding tokens
        counts = mask.sum(dim=1).clamp(min=1)
        # get the average embedding for each sentence
        averaged = summed / counts
        hidden = torch.relu(self.fc1(averaged))
        hidden = self.dropout(hidden)
        return self.fc2(hidden)

Key Training Techniques


Optimizer: Adam
 - Why: Automatically adjusts learning speed so training goes smoothly

Learning Rate: 5e-4
 - Why: Tested a few values. This gave the best accuracy without unstable jumps

Weight decay (L2): 5e-4 (0.0005)
 - Why: Penalizes complex patterns to stop the model from memorizing the small dataset (240 samples).

Batch size: 16
 - Why: Processes 16 samples per step. Small batches keep learning steady.

Epochs: 30
 - Why: How many times the model sees all data. 30 passes was the sweet spot between underfitting and overfitting.

Loss function: Weighted CrossEntropy
 - Why: Fixes class imbalance (180 vs 60) by giving more weight to errors on the minority class.

Dropout: 0.6
 - Why: Turns off 60% of neurons randomly each step so the model doesn't over-rely on specific features.

Random seed: 42
 - Why: Locks all random choices so your code gives identical results every run.

In [ ]:
#build a vocabulary from the training text
def train_one_model(x_tr, y_tr, epochs=30, batch_size=16, lr=5e-4, weight_decay=5e-4, vocab=None):
    if vocab is None:
        vocab = build_vocab(x_tr)
# calculate class weights to handle imbalanced data
    class_counts = y_tr.value_counts()
    class_weights = torch.tensor(
        [1.0 / class_counts[0], 1.0 / class_counts[1]], dtype=torch.float
    )
    class_weights = class_weights / class_weights.sum() * 2  # normalize, keep loss scale sane
# initialize the model, the optimizer in this case adam, and the loss function using the weights
    model = NeuralClassifier(vocab_size=len(vocab))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
# set up the dataset and data loader to feed training data in shuffled batches
    train_ds = ReviewDataset(x_tr, y_tr, vocab)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)

# Put model in training mode (enables dropout) and start the training loop
    model.train()
    for epoch in range(epochs):
        for xb, yb in train_dl:
            optimizer.zero_grad() # clear old gradients
            logits = model(xb) # make predictions
            loss = loss_fn(logits, yb) # calculate the error
            loss.backward() # calculate new gradients
            optimizer.step() # update the model weights
# return the fully trained model and its matching vocabulary
    return model, vocab

def predict(model, vocab, texts, batch_size=64):
    # put model in evaluation mode this turns off droput and sets up the data loader
    model.eval()
    ds = ReviewDataset(texts, None, vocab)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate)
    preds = []
    # disable gadient calculation to save memroy and speed up predictions
    with torch.no_grad():
        for xb, _ in dl:
            logits = model(xb) # get raw predicitons and pick the class with the highest score
            preds.extend(logits.argmax(dim=1).tolist())
# return the final list of predicted class labels
    return preds

In [ ]:
#set up 5 fold cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

#prepare lists to collect accuracy per fold and all combined true/predicted labels
fold_accuracies = []
cv_true, cv_pred = [], []

# loop through each of the 5 cross validation folds
for fold, (tr_idx, val_idx) in enumerate(cv.split(x_train, y_train)):
    x_tr, y_tr = x_train.iloc[tr_idx], y_train.iloc[tr_idx]
    x_val, y_val = x_train.iloc[val_idx], y_train.iloc[val_idx]

#train a new model for this fold and get predicitons on the validation set
    fold_model, fold_vocab = train_one_model(x_tr, y_tr)
    fold_preds = predict(fold_model, fold_vocab, x_val)

# calculate accuracy for this fold, store predicitions, and print the score
    acc = accuracy_score(y_val, fold_preds)
    fold_accuracies.append(acc)
    cv_true.extend(y_val.tolist())
    cv_pred.extend(fold_preds)
    print(f"Fold {fold}: accuracy = {acc:.3f}")

#calculate average accuracy and overall confusion matrix across all 5 folds
cv_accuracy = float(np.mean(fold_accuracies))
cv_matrix = confusion_matrix(cv_true, cv_pred, labels=[0, 1])

#print final overall results
print(f"\n5-Fold Cross-Validation Accuracy: {cv_accuracy:.3f}")
print("Cross-Validation Confusion Matrix (rows = true, cols = predicted, order = [negative, positive]):")
for row in cv_matrix:
    print(row)

Fold 0: accuracy = 0.562
Fold 1: accuracy = 0.729
Fold 2: accuracy = 0.604
Fold 3: accuracy = 0.708
Fold 4: accuracy = 0.667

5-Fold Cross-Validation Accuracy: 0.654
Cross-Validation Confusion Matrix (rows = true, cols = predicted, order = [negative, positive]):
[28 32]
[ 51 129]


In [ ]:
#train the final model using all available training data
final_model, final_vocab = train_one_model(x_train, y_train)

#generate predicitons
public_predictions = predict(final_model, final_vocab, x_public)
public_predictions = np.array(public_predictions).astype(int)

#calculate overall accuracy and build the confusion matrix
public_accuracy = accuracy_score(y_public, public_predictions)
public_matrix = confusion_matrix(y_public, public_predictions, labels=[0, 1])

#print total test accuracy and display the confusion matrix row by row
print(f"Public Test Total Accuracy: {public_accuracy:.3f}")
print("Public Test Confusion Matrix (rows = true, columns = predicted, order = [negative, positive]):")
for row in public_matrix:
    print(row)

print("\n Public Test Classification Report:")
print(classification_report(y_public, public_predictions, target_names=["Negative", "Positive"]))

Public Test Total Accuracy: 0.598
Public Test Confusion Matrix (rows = true, columns = predicted, order = [negative, positive]):
[108  92]
[ 69 131]

 Public Test Classification Report:
              precision    recall  f1-score   support

    Negative       0.61      0.54      0.57       200
    Positive       0.59      0.66      0.62       200

    accuracy                           0.60       400
   macro avg       0.60      0.60      0.60       400
weighted avg       0.60      0.60      0.60       400



In [102]:
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Save model weights
torch.save(final_model.state_dict(), CHECKPOINT_DIR / "model_state_dict.pt")

# Save the vocabulary — required to re-encode raw text the same way at inference time
with open(CHECKPOINT_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(final_vocab, f)

# Save metadata / architecture hyperparameters so the model can be reconstructed exactly
metadata = {
    "model": "Neural Bag-of-Embeddings (Deep Averaging Network): trainable embedding + mean pool + MLP",
    "embed_dim": 150,
    "hidden_dim": 128,
    "dropout": 0.6,
    "num_classes": 2,
    "vocab_size": len(final_vocab),
    "min_freq": MIN_FREQ,
    "epochs": 30,
    "batch_size": 16,
    "learning_rate": 5e-4,
    "weight_decay": 5e-4,
    "optimizer": "Adam",
    "loss": "class-weighted CrossEntropyLoss",
    "random_state": SEED,
    "train_rows": int(len(train)),
    "cv_accuracy": cv_accuracy,
    "public_test_accuracy": float(public_accuracy),
}
with open(CHECKPOINT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Checkpoint saved to:", CHECKPOINT_DIR / "model_state_dict.pt")
print("Vocabulary saved to:", CHECKPOINT_DIR / "vocab.json")
print("Metadata saved to:", CHECKPOINT_DIR / "metadata.json")

Checkpoint saved to: f:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main\model_checkpoint\model_state_dict.pt
Vocabulary saved to: f:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main\model_checkpoint\vocab.json
Metadata saved to: f:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main\model_checkpoint\metadata.json


In [ ]:
# create a pandas DataFrame containing the sample IDs and their predicted labels
public_submission = pd.DataFrame({
    "id": public_test["id"],
    "predicted_label": public_predictions,
})

# savve the dataframe as a csv file
public_submission.to_csv(ROOT / "public_test_predictions.csv", index=False)

#print the confirmation message and display the first 5 rows of the submission file
print("Prediction file saved to:", ROOT / "public_test_predictions.csv")
public_submission.head()

Prediction file saved to: f:\CYSE 650 - Ai Tools and\HW\HW2\CYSE499_650_Summer2026-main\public_test_predictions.csv


,id,predicted_label
0,pos_cv696_29740,1
1,pos_cv669_22995,1
2,neg_cv963_7208,1
3,pos_cv182_7281,0
4,pos_cv162_10424,1


In [ ]:
#inference only sanity check
with open(CHECKPOINT_DIR / "vocab.json", "r", encoding="utf-8") as f:
    loaded_vocab = json.load(f)
with open(CHECKPOINT_DIR / "metadata.json", "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

#recreate the model architecture using the loaded metadata
reloaded_model = NeuralClassifier(
    vocab_size=loaded_meta["vocab_size"],
    embed_dim=loaded_meta["embed_dim"],
    hidden_dim=loaded_meta["hidden_dim"],
    dropout=loaded_meta["dropout"],
)

#load the saved model weights and set the model to evaluation mode
reloaded_model.load_state_dict(torch.load(CHECKPOINT_DIR / "model_state_dict.pt"))
reloaded_model.eval()

#make predictions on the test set with the reloaded model and print its accuracy
reloaded_preds = predict(reloaded_model, loaded_vocab, x_public)
reloaded_accuracy = accuracy_score(y_public, reloaded_preds)
print(f"Public Test Accuracy From Reloaded Checkpoint: {reloaded_accuracy:.3f}")

#verify the reloaded models predictions match the origianl run exactly
assert reloaded_preds == list(public_predictions), "Reloaded Checkpoint Predictions do not match the original run"
print("Reloaded Checkpoint Predictions match the original run exactly")

Public Test Accuracy From Reloaded Checkpoint: 0.598
Reloaded Checkpoint Predictions match the original run exactly


Summary

for this assignment, specifically part 1, I built a sentiment classifier using a Deep Averaging Network. This is basically trainable word embeddings that are averaged across each review, and then fed into a small multilayer perceptron or MLP with dropout. I used Adam with 5e-4 learning rate, 5e-4 weight decay, a batch size of 16, and 30 epochs, which i picked after running a quick manual grid search over 5-fold cross validation accuracy.

Because the training set was so small and imbalanced (240 total reviews: 180 of which were positive, and 60 negative), I used class weighted loss, dropout, weight decay, a low capacity model, and a minimum frequency cutoff for the vocab. Any word that appeared fewer than 2 times in training would get mapped to <unk>, which also helped the model handle unknwon words during its evaluation period.

The 5-fold cross validation accuracy came out to 0.654, with a combined confusion matrix of [28, 32], [51, 129] the rows/columns = negative, positive. Across all folds through, it caught 28 of 60 negative reviews which equates to 47% recall and 129 of 180 positive reviews which is a 72% recall.

On the 400 review public test set, the accuracy was 0.598 with a confusion matrix of [108, 92], [69, 131]. Essentially giving them a 54% recall on negatives and 66% recall on the positives. This is a much more balanced split than my inital attempt with a non-neural baseline where i used TF-IDF with logistic regression. While that baseline hit a similar test accuracy of 58.5%, it caught 98% of positives and only 18% of negatives, meaning it was basically just defaulting to "positive". The neural model is way less biased now, even if the overall accuracy looks similar.

The gap between CV accuracy (0.654) and public test accuracy (0.598) is also much smaller than the baselines drop from 0.787 to 0.585, thus showing less overfitting. Still there is an obvious generalization gap when learning embeddings from scratch on only 240 examples. If i had more time or compute, i would initialize the embedding layer with pretrained vectors like GloVe instead of training them from scratch starting with decent representation rather than trying to learn everything from 240 reviews would probably close a good chunk of that gap.

Use of AI

I specifically used AI for debugging some code. I was having issues with it running, it couldnt access the .venv as well as some of the imports. I also used for some formatting of the build code and a few error messages. On my own i modeled my approach by building an outline, finalizing my approach, analysis of the outputs, and my written interpretation. AI was used ont he section called "Key Training Techniques" as i needed help formatting it to be a table to read easily.